# Sample4Geo Fine-tuning — 연남동 데이터 (도로뷰 + 드론뷰 + 위성뷰)

**학습 구성**
- Query(쿼리): 도로뷰 파노라마(`panorama/`) 또는 드론뷰(`drone/`)
- Reference(레퍼런스): 위성뷰(`satellite/`)
- 손실함수: InfoNCE (Symmetric Contrastive Learning)

**Google Drive 데이터 구조**
```
MyDrive/
└── yeongnamdong_dataset/
    ├── satellite/
    │   ├── meta.csv          # index, file, lat, lng
    │   └── sat_00000.png ...
    ├── panorama/
    │   ├── meta.csv          # index, file, lat, lng
    │   └── pano_00000.png ...
    └── drone/
        ├── meta.csv          # index, file, lat, lng, pano_id, azimuth
        └── drone_00000.png ...
```

> **가중치**: [Sample4Geo 공식 Drive](https://drive.google.com/drive/folders/1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA)에서
> `convnext_base` CVUSA 또는 CVACT `.pth` 파일 다운로드 후 Drive에 업로드하세요.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 데이터 압축 해제 (Drive → Colab 로컬 SSD)

Drive에서 직접 이미지를 읽으면 I/O가 느립니다.  
ZIP을 Colab의 로컬 디스크(`/content/`)에 풀면 학습 속도가 크게 향상됩니다.

> **한 번만 실행하면 됩니다.** 런타임이 초기화되면 다시 실행 필요.

In [ ]:
import os, time

# ── Drive 내 ZIP 경로 (필요시 수정) ────────────────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/yeongnamdong_dataset.zip'
OUT_DIR  = '/content/yeongnamdong_dataset'

if os.path.exists(OUT_DIR):
    print(f'이미 압축 해제됨: {OUT_DIR}')
else:
    print('압축 해제 중... (2~5분 소요)')
    t0 = time.time()
    os.makedirs(OUT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d /content/
    print(f'완료! ({time.time()-t0:.0f}초)')

# 파일 수 확인
for folder in ['satellite', 'panorama', 'drone']:
    n = len([f for f in os.listdir(f'{OUT_DIR}/{folder}') if f.endswith('.png')])
    print(f'  {folder}: {n}개')

압축 해제 중... (2~5분 소요)
완료! (65초)
  satellite: 1642개
  panorama: 877개
  drone: 7074개


## 2. Sample4Geo 클론 및 패키지 설치

In [ ]:
import os

if not os.path.exists('/content/Sample4Geo'):
    !git clone https://github.com/Skyy93/Sample4Geo.git /content/Sample4Geo
else:
    print('Sample4Geo already cloned')

!pip install -q -r /content/Sample4Geo/requirements.txt
print('설치 완료')

Cloning into '/content/Sample4Geo'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 119 (delta 55), reused 40 (delta 40), pack-reused 51 (from 1)
Receiving objects: 100% (119/119), 770.78 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (72/72), done.
설치 완료


## 3. 공통 임포트

In [ ]:
import sys
import time
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
from dataclasses import dataclass, field
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from scipy.spatial import KDTree
from tqdm import tqdm
from transformers import (
    get_cosine_schedule_with_warmup,
    get_polynomial_decay_schedule_with_warmup,
    get_constant_schedule_with_warmup,
)

sys.path.insert(0, '/content/Sample4Geo')
from sample4geo.model import TimmModel
from sample4geo.trainer import train, predict
from sample4geo.transforms import get_transforms_train, get_transforms_val
from sample4geo.loss import InfoNCE
from sample4geo.utils import setup_system
from sample4geo.evaluate.cvusa_and_cvact import calculate_scores

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

PyTorch: 2.11.0+cu128
GPU: Tesla T4


## 4. 설정 (여기만 수정)

| 항목 | 설명 |
|------|------|
| `data_folder` | Drive 내 `yeongnamdong_dataset/` 경로 |
| `query_types` | 학습에 사용할 뷰 타입 목록 (`panorama`, `drone`, 또는 둘 다) |
| `checkpoint_start` | 사전학습 가중치 `.pth` 경로 |
| `save_path` | 체크포인트 저장 경로 |

In [ ]:
@dataclass
class Config:

    # ── 경로 ───────────────────────────────────────────────────────────────
    data_folder:      str  = '/content/yeongnamdong_dataset'
    checkpoint_start: str  = '/content/drive/MyDrive/weights/sample4geo_cvusa.pth'
    save_path:        str  = '/content/drive/MyDrive/finetune_output'

    # ── 학습 쿼리 뷰 선택 ──────────────────────────────────────────────────
    query_types: object = field(default_factory=lambda: ['panorama', 'drone'])

    # ── 모델 ───────────────────────────────────────────────────────────────
    model:    str = 'convnext_base.fb_in22k_ft_in1k_384'
    img_size: int = 384

    # ── 데이터 분할 ─────────────────────────────────────────────────────────
    val_split: float = 0.2
    seed:      int   = 42

    # ── 학습 ───────────────────────────────────────────────────────────────
    mixed_precision: bool  = True
    epochs:          int   = 10
    batch_size:      int   = 16
    verbose:         bool  = True
    gpu_ids:         tuple = (0,)

    # ── 데이터 증강 ─────────────────────────────────────────────────────────
    prob_flip:   float = 0.5
    prob_rotate: float = 0.75

    # ── 평가 ───────────────────────────────────────────────────────────────
    batch_size_eval:    int  = 32
    eval_every_n_epoch: int  = 2
    normalize_features: bool = True

    # ── 옵티마이저 ──────────────────────────────────────────────────────────
    lr:                float = 1e-4
    scheduler:         str   = 'cosine'
    warmup_epochs:     int   = 1
    lr_end:            float = 1e-6
    clip_grad:         float = 100.
    decay_exclue_bias: bool  = False
    grad_checkpointing:bool  = False

    # ── 손실함수 ────────────────────────────────────────────────────────────
    label_smoothing: float = 0.15   # 0.1 → 0.15 (0.3은 batch=16에서 너무 강함)

    # ── 시스템 ─────────────────────────────────────────────────────────────
    device:              str  = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers:         int  = 2
    cudnn_benchmark:     bool = True
    cudnn_deterministic: bool = False


config = Config()
os.makedirs(config.save_path, exist_ok=True)

if isinstance(config.query_types, str):
    config.query_types = [config.query_types]

print('학습 쿼리 타입:', config.query_types)
print('데이터 경로:  ', config.data_folder)
print('가중치 경로:  ', config.checkpoint_start)
print('저장 경로:    ', config.save_path)
print('device:       ', config.device)

학습 쿼리 타입: ['panorama', 'drone']
데이터 경로:   /content/yeongnamdong_dataset
가중치 경로:   /content/drive/MyDrive/weights/sample4geo_cvusa.pth
저장 경로:     /content/drive/MyDrive/finetune_output
device:        cuda


## 5. 커스텀 Dataset

### 설계 원칙

**위성 위치 기준 train/val 분리**
- 같은 위성에 매칭되는 모든 쿼리(파노라마 + 드론 전 방위각)를 동일 split에 배정
- 위성이 train/val 사이에 겹치지 않아 데이터 누수 없음

**InfoNCE False Negative 방지 (핵심)**
- 위성 1개당 드론뷰가 ~20장 매칭됨
- 배치 내에 같은 위성을 참조하는 쿼리가 2개 이상 → InfoNCE가 잘못된 negative 처리
- **해결**: epoch마다 위성당 쿼리 1개만 무작위 선택 (`_resample()`)
  → 배치 내 모든 reference가 서로 다른 위성 보장

```
위성 A ─┬─ panorama_0042   epoch 1 → panorama 선택
        ├─ drone_az018     epoch 2 → drone az018 선택
        ├─ drone_az036     epoch 3 → drone az036 선택  (매 epoch 다른 뷰 = 증강 효과)
        └─ drone_az054 ...
```

In [ ]:
import random
from collections import defaultdict


def _load_meta(data_folder, view_type):
    """meta.csv 로드 → [(img_path, lat, lng), ...] 반환"""
    df = pd.read_csv(os.path.join(data_folder, view_type, 'meta.csv'))
    return [
        (os.path.join(data_folder, row['file']), float(row['lat']), float(row['lng']))
        for _, row in df.iterrows()
    ]


def _build_split_pairs(data_folder, query_types, val_split=0.2, seed=42):
    """
    위성 위치 기준으로 train/val 분리.

    같은 위성에 매칭된 파노라마·드론 전체를 동일 split에 배정하므로
    위성 위치가 train/val 사이에 겹치지 않는다.

    반환값
    ------
    train_pairs : [(q_path, s_path, sat_idx), ...]
    val_pairs   : [(q_path, s_path, sat_idx), ...]
    sat_records : [(s_path, lat, lng), ...]   전체 위성 목록
    """
    sat_records = _load_meta(data_folder, 'satellite')
    sat_coords  = np.array([(r[1], r[2]) for r in sat_records])
    tree        = KDTree(sat_coords)

    all_pairs = []
    for qtype in query_types:
        q_records = _load_meta(data_folder, qtype)
        q_coords  = np.array([(r[1], r[2]) for r in q_records])
        _, sat_indices = tree.query(q_coords)
        for qi, si in enumerate(sat_indices):
            all_pairs.append((q_records[qi][0], sat_records[si][0], int(si)))

    # 위성 인덱스 기준 train/val 분리
    unique_sats = list({p[2] for p in all_pairs})
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_sats)
    n_val      = int(len(unique_sats) * val_split)
    val_sats   = set(unique_sats[:n_val])
    train_sats = set(unique_sats[n_val:])

    train_pairs = [p for p in all_pairs if p[2] in train_sats]
    val_pairs   = [p for p in all_pairs if p[2] in val_sats]
    return train_pairs, val_pairs, sat_records


def _filter_missing(images, labels, tag=''):
    """존재하지 않는 파일 경로를 제거하고 경고 출력"""
    valid = [(p, l) for p, l in zip(images, labels) if os.path.exists(p)]
    n_removed = len(images) - len(valid)
    if n_removed:
        print(f'  [경고]{tag} 파일 없음 {n_removed}개 제외 (전체 {len(images)}개 중)')
    if valid:
        imgs, lbls = zip(*valid)
        return list(imgs), list(lbls)
    return [], []


class YeongnamdongTrainDataset(Dataset):
    """
    학습용 Dataset.

    위성당 쿼리 1개 보장 (_resample)
    ─────────────────────────────────
    같은 위성에 매칭된 파노라마·드론 여러 장을 sat_idx로 그룹화.
    epoch 시작 시 _resample() → 그룹마다 1개 무작위 선택.

    효과
    ────
    · 배치 내 reference 가 모두 다른 위성 → InfoNCE false negative 없음
    · 매 epoch 다른 뷰가 선택 → 아지무스 다양성이 데이터 증강으로 작용
    """

    def __init__(self, data_folder, query_types,
                 transforms_query=None, transforms_reference=None,
                 prob_flip=0.0, prob_rotate=0.0,
                 val_split=0.2, seed=42):
        super().__init__()
        train_pairs, _, _ = _build_split_pairs(
            data_folder, query_types, val_split=val_split, seed=seed)

        # 존재하지 않는 쿼리 파일 사전 제거
        all_q  = [p[0] for p in train_pairs]
        all_lb = [p[2] for p in train_pairs]
        valid_q, valid_lb = _filter_missing(all_q, all_lb, tag='[train query]')
        train_pairs = [(q, train_pairs[i][1], lb)
                       for i, (q, lb) in enumerate(zip(valid_q, valid_lb))]

        # sat_idx → [pair, ...]  (파노라마 + 드론 전 방위각 모두 포함)
        self._groups  = defaultdict(list)
        for p in train_pairs:
            self._groups[p[2]].append(p)
        self._sat_keys = list(self._groups.keys())

        self.transforms_query     = transforms_query
        self.transforms_reference = transforms_reference
        self.prob_flip   = prob_flip
        self.prob_rotate = prob_rotate

        self._resample()   # 초기 샘플 생성

        n_total = sum(len(v) for v in self._groups.values())
        print(f'  학습 위성 수:         {len(self._sat_keys)}')
        print(f'  전체 쿼리-위성 쌍 수: {n_total}  '
              f'(epoch당 {len(self._sat_keys)}개 사용)')

    def _resample(self):
        """epoch 시작마다 호출 — 위성당 쿼리 1개 무작위 선택 후 셔플"""
        self._epoch_pairs = [
            random.choice(self._groups[k]) for k in self._sat_keys
        ]
        random.shuffle(self._epoch_pairs)

    def __len__(self):
        return len(self._epoch_pairs)

    def __getitem__(self, index):
        q_path, s_path, sat_idx = self._epoch_pairs[index]

        q_img = cv2.imread(q_path)
        s_img = cv2.imread(s_path)
        if q_img is None:
            raise FileNotFoundError(f'쿼리 이미지 읽기 실패: {q_path}')
        if s_img is None:
            raise FileNotFoundError(f'위성 이미지 읽기 실패: {s_path}')
        q_img = cv2.cvtColor(q_img, cv2.COLOR_BGR2RGB)
        s_img = cv2.cvtColor(s_img, cv2.COLOR_BGR2RGB)

        if np.random.random() < self.prob_flip:
            q_img = cv2.flip(q_img, 1)
            s_img = cv2.flip(s_img, 1)

        if self.transforms_query:
            q_img = self.transforms_query(image=q_img)['image']
        if self.transforms_reference:
            s_img = self.transforms_reference(image=s_img)['image']

        if np.random.random() < self.prob_rotate:
            r     = np.random.choice([1, 2, 3])
            s_img = torch.rot90(s_img, k=r, dims=(1, 2))
            _, _, w = q_img.shape
            q_img = torch.roll(q_img, shifts=-(w // 4 * r), dims=2)

        return q_img, s_img, torch.tensor(sat_idx, dtype=torch.long)


class YeongnamdongEvalDataset(Dataset):
    """
    평가용.
    img_type='query'     → 검증 쿼리 (드론 전 방위각 모두 포함), 레이블 = sat_idx
    img_type='reference' → 전체 위성 갤러리 1642장,              레이블 = sat_idx
    """

    def __init__(self, data_folder, img_type, query_types=None,
                 transforms=None, val_split=0.2, split='val', seed=42):
        super().__init__()
        self.transforms = transforms

        if img_type == 'reference':
            sat_records  = _load_meta(data_folder, 'satellite')
            images  = [r[0] for r in sat_records]
            labels  = list(range(len(sat_records)))

        elif img_type == 'query':
            train_pairs, val_pairs, _ = _build_split_pairs(
                data_folder, query_types or ['panorama'],
                val_split=val_split, seed=seed)

            if split == 'val':
                pairs = val_pairs
            else:
                pairs = train_pairs + val_pairs   # 전체 평가

            images = [p[0] for p in pairs]
            labels = [p[2] for p in pairs]

        else:
            raise ValueError("img_type must be 'query' or 'reference'")

        # 존재하지 않는 파일 사전 제거
        self.images, self.labels = _filter_missing(
            images, labels, tag=f'[{img_type}/{split}]')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        path = self.images[index]
        img  = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f'이미지 읽기 실패: {path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms:
            img = self.transforms(image=img)['image']
        return img, torch.tensor(self.labels[index], dtype=torch.long)


print('Dataset 클래스 정의 완료')

Dataset 클래스 정의 완료


## 6. 모델 로드 및 사전학습 가중치 적용

In [ ]:
setup_system(seed=config.seed,
             cudnn_benchmark=config.cudnn_benchmark,
             cudnn_deterministic=config.cudnn_deterministic)

model       = TimmModel(config.model, pretrained=True, img_size=config.img_size)
data_config = model.get_config()
mean, std   = data_config['mean'], data_config['std']

image_size_sat    = (config.img_size, config.img_size)
new_width         = config.img_size * 2
new_height        = round((224 / 1232) * new_width)
image_size_ground = (new_height, new_width)

print(f'위성 크기:  {image_size_sat}')
print(f'쿼리 크기:  {image_size_ground}')
print(f'mean={mean}  std={std}')

if config.grad_checkpointing:
    model.set_grad_checkpointing(True)

if os.path.isfile(config.checkpoint_start):
    print(f'\n사전학습 가중치 로드: {config.checkpoint_start}')
    state = torch.load(config.checkpoint_start, map_location='cpu')
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f'  누락: {len(missing)}  예상치 못한 키: {len(unexpected)}')
else:
    print(f'[경고] 가중치 파일 없음 → timm ImageNet 가중치만 사용')

model = model.to(config.device)
print('\n모델 준비 완료')

model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

위성 크기:  (384, 384)
쿼리 크기:  (140, 768)
mean=(0.485, 0.456, 0.406)  std=(0.229, 0.224, 0.225)
[경고] 가중치 파일 없음 → timm ImageNet 가중치만 사용

모델 준비 완료


## 7. 데이터로더 구성

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── 강화된 transforms 인라인 정의 ────────────────────────────────────────────
# 위성 이미지에 RandomResizedCrop 사용 금지:
#   크롭 시 GPS 매칭 위치가 크롭 영역 밖으로 벗어나 학습 신호가 파괴됨

def _sat_transforms_train(image_size, mean, std):
    return A.Compose([
        A.ImageCompression(quality=(90, 100), p=0.5),
        A.Resize(image_size[0], image_size[1],
                 interpolation=cv2.INTER_LINEAR_EXACT, p=1.0),
        A.ColorJitter(
            brightness=0.20, contrast=0.20,
            saturation=0.20, hue=0.10, p=0.6),
        A.OneOf([A.AdvancedBlur(p=1.0), A.Sharpen(p=1.0)], p=0.3),
        A.OneOf([
            A.GridDropout(ratio=0.4, p=1.0),
            A.CoarseDropout(
                max_holes=25,
                max_height=int(0.2 * image_size[0]),
                max_width=int(0.2 * image_size[0]),
                min_holes=10,
                min_height=int(0.1 * image_size[0]),
                min_width=int(0.1 * image_size[0]),
                p=1.0),
        ], p=0.4),
        A.Normalize(mean, std),
        ToTensorV2(),
    ])


def _gnd_transforms_train(image_size, mean, std):
    return A.Compose([
        A.ImageCompression(quality=(90, 100), p=0.5),
        A.Resize(image_size[0], image_size[1],
                 interpolation=cv2.INTER_LINEAR_EXACT, p=1.0),
        A.ColorJitter(
            brightness=0.20, contrast=0.20,
            saturation=0.20, hue=0.10, p=0.6),
        A.OneOf([A.AdvancedBlur(p=1.0), A.Sharpen(p=1.0)], p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.OneOf([
            A.GridDropout(ratio=0.5, p=1.0),
            A.CoarseDropout(
                max_holes=25,
                max_height=int(0.2 * image_size[0]),
                max_width=int(0.2 * image_size[0]),
                min_holes=10,
                min_height=int(0.1 * image_size[0]),
                min_width=int(0.1 * image_size[0]),
                p=1.0),
        ], p=0.4),
        A.Normalize(mean, std),
        ToTensorV2(),
    ])


def _sat_transforms_val(image_size, mean, std):
    return A.Compose([
        A.Resize(image_size[0], image_size[1],
                 interpolation=cv2.INTER_LINEAR_EXACT, p=1.0),
        A.Normalize(mean, std),
        ToTensorV2(),
    ])


def _gnd_transforms_val(image_size, mean, std):
    return A.Compose([
        A.Resize(image_size[0], image_size[1],
                 interpolation=cv2.INTER_LINEAR_EXACT, p=1.0),
        A.Normalize(mean, std),
        ToTensorV2(),
    ])


sat_tf_train = _sat_transforms_train(image_size_sat,    mean, std)
gnd_tf_train = _gnd_transforms_train(image_size_ground, mean, std)
sat_tf_val   = _sat_transforms_val(image_size_sat,    mean, std)
gnd_tf_val   = _gnd_transforms_val(image_size_ground, mean, std)

print('Transforms 정의 완료 (v3 — RandomResizedCrop 제거, 증강 강도 조정)')

# ── 데이터로더 구성 ───────────────────────────────────────────────────────────
train_dataset = YeongnamdongTrainDataset(
    data_folder=config.data_folder,
    query_types=config.query_types,
    transforms_query=gnd_tf_train,
    transforms_reference=sat_tf_train,
    prob_flip=config.prob_flip,
    prob_rotate=config.prob_rotate,
    val_split=config.val_split,
    seed=config.seed,
)
train_dataloader = DataLoader(
    train_dataset, batch_size=config.batch_size,
    shuffle=True, num_workers=config.num_workers,
    pin_memory=True, drop_last=True,
)

query_dataset_val = YeongnamdongEvalDataset(
    data_folder=config.data_folder, img_type='query',
    query_types=config.query_types,
    transforms=gnd_tf_val,
    val_split=config.val_split, split='val', seed=config.seed,
)
query_dataloader_val = DataLoader(
    query_dataset_val, batch_size=config.batch_size_eval,
    shuffle=False, num_workers=config.num_workers, pin_memory=True,
)

reference_dataset_val = YeongnamdongEvalDataset(
    data_folder=config.data_folder, img_type='reference',
    transforms=sat_tf_val,
)
reference_dataloader_val = DataLoader(
    reference_dataset_val, batch_size=config.batch_size_eval,
    shuffle=False, num_workers=config.num_workers, pin_memory=True,
)

print(f'\n학습 샘플:     {len(train_dataset):>6}  (쿼리 타입: {config.query_types})')
print(f'검증 쿼리:     {len(query_dataset_val):>6}')
print(f'검증 레퍼런스: {len(reference_dataset_val):>6}  (전체 위성)')

Transforms 정의 완료 (v3 — RandomResizedCrop 제거, 증강 강도 조정)


/tmp/ipykernel_568/2094743014.py:10: UserWarning: Argument(s) 'quality' are not valid for transform ImageCompression
  A.ImageCompression(quality=(90, 100), p=0.5),
/tmp/ipykernel_568/2094743014.py:19: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width' are not valid for transform CoarseDropout
  A.CoarseDropout(
/tmp/ipykernel_568/2094743014.py:35: UserWarning: Argument(s) 'quality' are not valid for transform ImageCompression
  A.ImageCompression(quality=(90, 100), p=0.5),
/tmp/ipykernel_568/2094743014.py:42: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
/tmp/ipykernel_568/2094743014.py:45: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width' are not valid for transform CoarseDropout
  A.CoarseDropout(


  학습 위성 수:         706
  전체 쿼리-위성 쌍 수: 6446  (epoch당 706개 사용)

학습 샘플:        706  (쿼리 타입: ['panorama', 'drone'])
검증 쿼리:       1505
검증 레퍼런스:   1642  (전체 위성)


## 8. 손실함수 / 옵티마이저 / 스케줄러

In [ ]:
loss_fn       = torch.nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
loss_function = InfoNCE(loss_function=loss_fn, device=config.device)
scaler        = GradScaler(init_scale=2.**10) if config.mixed_precision else None

if config.decay_exclue_bias:
    no_decay = ['bias', 'LayerNorm.bias']
    params   = list(model.named_parameters())
    opt_params = [
        {'params': [p for n, p in params if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in params if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(opt_params, lr=config.lr)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)

train_steps  = len(train_dataloader) * config.epochs
warmup_steps = len(train_dataloader) * config.warmup_epochs

if config.scheduler == 'polynomial':
    scheduler = get_polynomial_decay_schedule_with_warmup(
        optimizer, num_training_steps=train_steps,
        lr_end=config.lr_end, power=1.5, num_warmup_steps=warmup_steps)
elif config.scheduler == 'cosine':
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_training_steps=train_steps, num_warmup_steps=warmup_steps)
elif config.scheduler == 'constant':
    scheduler = get_constant_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps)
else:
    scheduler = None

print(f'스케줄러: {config.scheduler}  |  warmup {config.warmup_epochs} epoch ({warmup_steps} step)')
print(f'전체:    {config.epochs} epoch ({train_steps} step)')

스케줄러: cosine  |  warmup 1 epoch (44 step)
전체:    10 epoch (440 step)


/tmp/ipykernel_568/809566557.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler        = GradScaler(init_scale=2.**10) if config.mixed_precision else None


## 9. 평가 함수

In [ ]:
def run_eval(config, model, query_dl, reference_dl, ranks=[1, 5, 10], step_size=500):
    """R@K 평가 (query 레이블 == reference 레이블이면 정답)"""
    print('  Feature 추출 중...')
    ref_features,   ref_labels   = predict(config, model, reference_dl)
    query_features, query_labels = predict(config, model, query_dl)

    print('  Score 계산 중...')
    r1 = calculate_scores(
        query_features, ref_features,
        query_labels,   ref_labels,
        step_size=step_size, ranks=ranks,
    )
    del ref_features, ref_labels, query_features, query_labels
    gc.collect()
    return r1


print('run_eval 정의 완료')

run_eval 정의 완료


## 10. 학습 루프

In [ ]:
best_score = 0.0
run_id     = time.strftime('%Y%m%d_%H%M%S')
ckpt_dir   = os.path.join(config.save_path, run_id)
os.makedirs(ckpt_dir, exist_ok=True)
print(f'체크포인트 저장 경로: {ckpt_dir}')

for epoch in range(1, config.epochs + 1):
    print(f'\n{"-"*28}[Epoch {epoch}/{config.epochs}]{"-"*28}')

    # ── epoch 시작마다 위성당 쿼리 1개 재선택 ──────────────────────────────
    # 같은 위성에 매칭된 파노라마/드론 중 이번 epoch에 사용할 뷰를 무작위 선택.
    # 배치 내 reference가 모두 다른 위성이 되어 InfoNCE false negative가 없음.
    train_dataset._resample()

    train_loss = train(
        config, model,
        dataloader=train_dataloader,
        loss_function=loss_function,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
    )
    print(f'loss={train_loss:.4f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

    if epoch % config.eval_every_n_epoch == 0 or epoch == config.epochs:
        print(f'{"-"*20}[Evaluate]{"-"*20}')
        r1 = run_eval(config, model,
                      query_dataloader_val, reference_dataloader_val)

        if r1 > best_score:
            best_score = r1
            save_file  = os.path.join(ckpt_dir, f'weights_e{epoch}_r1_{r1:.4f}.pth')
            torch.save(model.state_dict(), save_file)
            print(f'  ★ Best R@1={best_score:.4f} → {save_file}')

final_path = os.path.join(ckpt_dir, 'weights_final.pth')
torch.save(model.state_dict(), final_path)
print(f'\n학습 완료  |  Best R@1: {best_score:.4f}')
print(f'최종 가중치: {final_path}')

체크포인트 저장 경로: /content/drive/MyDrive/finetune_output/20260621_161249

----------------------------[Epoch 1/10]----------------------------


  0%|          | 0/44 [00:00<?, ?it/s]/content/Sample4Geo/sample4geo/trainer.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 44/44 [01:31<00:00,  2.08s/it, loss=2.6248, loss_avg=2.8455, lr=0.000100]


loss=2.8455  lr=1.00e-04

----------------------------[Epoch 2/10]----------------------------


100%|██████████| 44/44 [00:55<00:00,  1.27s/it, loss=2.5045, loss_avg=2.5645, lr=0.000097]


loss=2.5645  lr=9.70e-05
--------------------[Evaluate]--------------------
  Feature 추출 중...


...

loss=1.6342  lr=0.00e+00
--------------------[Evaluate]--------------------
  Feature 추출 중...


100%|██████████| 48/48 [00:17<00:00,  2.72it/s]


  Score 계산 중...


100%|██████████| 1505/1505 [00:00<00:00, 15732.69it/s]


Recall@1: 2.5249 - Recall@5: 13.1561 - Recall@10: 22.0598 - Recall@top1: 29.9668

학습 완료  |  Best R@1: 2.9236
최종 가중치: /content/drive/MyDrive/finetune_output/20260621_161249/weights_final.pth


## 11. 전체 데이터 최종 평가

In [ ]:
query_dataset_all = YeongnamdongEvalDataset(
    data_folder=config.data_folder, img_type='query',
    query_types=config.query_types,
    transforms=gnd_tf_val,
    val_split=config.val_split, split='all', seed=config.seed,
)
query_dataloader_all = DataLoader(
    query_dataset_all, batch_size=config.batch_size_eval,
    shuffle=False, num_workers=config.num_workers, pin_memory=True,
)

print(f'전체 평가 쿼리 수: {len(query_dataset_all)}')
print('\n[전체 데이터 최종 평가]')
r1_final = run_eval(
    config, model,
    query_dataloader_all, reference_dataloader_val,
)
print(f'최종 R@1: {r1_final:.4f}')

전체 평가 쿼리 수: 7951

[전체 데이터 최종 평가]
  Feature 추출 중...


  0%|          | 0/52 [00:00<?, ?it/s]/content/Sample4Geo/sample4geo/trainer.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
 64%|██████▍   | 159/249 [00:57<00:32,  2.77it/s]


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_568/308969949.py", line 196, in __getitem__
    raise FileNotFoundError(f'이미지 읽기 실패: {path}')
FileNotFoundError: 이미지 읽기 실패: /content/yeongnamdong_dataset/drone/drone_05419.png


## 12. 뷰 타입별 개별 평가 (도로뷰 vs 드론뷰 성능 비교)

In [ ]:
for vtype in config.query_types:
    print(f'\n[{vtype.upper()} 단독 평가]')
    ds = YeongnamdongEvalDataset(
        data_folder=config.data_folder, img_type='query',
        query_types=[vtype],
        transforms=gnd_tf_val,
        val_split=config.val_split, split='all', seed=config.seed,
    )
    dl = DataLoader(ds, batch_size=config.batch_size_eval,
                    shuffle=False, num_workers=config.num_workers, pin_memory=True)
    print(f'  쿼리 수: {len(ds)}')
    run_eval(config, model, dl, reference_dataloader_val)

## 13. Best 체크포인트 로드 및 재평가 (선택)

In [ ]:
BEST_WEIGHT_PATH = ''  # 예: '.../weights_e8_r1_0.8700.pth'

if BEST_WEIGHT_PATH and os.path.isfile(BEST_WEIGHT_PATH):
    print(f'로드: {BEST_WEIGHT_PATH}')
    model.load_state_dict(torch.load(BEST_WEIGHT_PATH, map_location=config.device))
    model.to(config.device)
    print('\n[Best 가중치 전체 평가]')
    run_eval(config, model, query_dataloader_all, reference_dataloader_val)
else:
    print('BEST_WEIGHT_PATH를 입력하세요.')